# 12 · Evaluation

Every pipeline ends by handing a trained model to an `Evaluator`. This guide
drives the evaluator directly, on three fast models fitted to MovieLens-100K,
and shows the `evaluation` section of a configuration next to each step. It
covers:

- the `Evaluator`: metrics × cutoffs in one pass, what `compute_results()`
  returns, and how a per-user vector becomes the number a run reports;
- full against sampled evaluation, with `uniform` and `popularity` negatives;
- `mask_seen`, and what a ranking looks like when training items stay in it;
- a tour of the metric families, including the ones that read side
  information, clusters or parameters;
- per-user results, `save_per_user`, `validation_metric` and
  `full_evaluation_on_report`;
- significance tests between models, with corrections for multiple testing.

Running whole pipelines is guide 10's subject; models are guide 9's.

**Requirements:** `pip install warprec jupyter`. Runs in about a minute on
MovieLens-100K (`guide_data.movielens_100k()` downloads and converts it exactly
as guide 1 shows).

In [1]:
import sys
from pathlib import Path

import narwhals as nw
import pandas as pd
import torch
import yaml
from loguru import logger

sys.path.insert(0, "..")
from guide_data import GENRES, movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/12-evaluation")
RUNS.mkdir(parents=True, exist_ok=True)
logger.disable("warprec")

## Data and models

Some metrics read more than interactions: `SRecall` reads item features,
the fairness metrics read user and item clusters. So the dataset carries the
19 genres as side information, users clustered by gender and items by their
first genre, built as guide 4 builds them.

In [2]:
items = pd.read_csv(folder / "items.tsv", sep="\t")
users = pd.read_csv(folder / "users.tsv", sep="\t")

genres = pd.DataFrame({"item_id": items.item_id})
for genre in GENRES:
    genres[genre] = items.genres.str.split("|").apply(lambda gs, g=genre: int(g in gs))
genres.to_csv(RUNS / "item-genres.tsv", sep="\t", index=False)

pd.DataFrame({"user_id": users.user_id, "cluster": users.gender}).to_csv(
    RUNS / "user-gender.tsv", sep="\t", index=False
)
pd.DataFrame(
    {"item_id": items.item_id, "cluster": items.genres.str.split("|").str[0]}
).to_csv(RUNS / "item-first-genre.tsv", sep="\t", index=False)

The test split is each user's latest 20% of interactions, read as implicit
feedback. The three models are closed-form, so building them is fitting them.

In [3]:
print(Path("configs/evaluation.yml").read_text())

# MovieLens-100K with genres as side information and two cluster files, split
# on each user's latest 20%. Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  dtypes:
    cluster_type: str
  side:
    local_path: ../runs/12-evaluation/item-genres.tsv
  clustering:
    user_local_path: ../runs/12-evaluation/user-gender.tsv
    item_local_path: ../runs/12-evaluation/item-first-genre.tsv
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  Pop: {}
  ItemKNN:
    k: 50
    similarity: cosine
  EASE:
    l2: 200.0
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall, Precision, HitRate]



In [4]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration
from warprec.utils.registry import model_registry

config = load_design_configuration("configs/evaluation.yml")
dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
models = {
    name: model_registry.get(
        name, params=params, info=dataset.info(), interactions=dataset.train_set
    )
    for name, params in config.models.items()
}
n_users, n_items = dataset.get_dims()
print(
    f"{n_users} users, {n_items} items, "
    f"{dataset.train_set.get_sparse().nnz} training and "
    f"{dataset.eval_set.get_sparse().nnz} test interactions"
)

943 users, 1613 items, 79619 training and 20295 test interactions


## The `Evaluator`

An `Evaluator` is built once from a list of metric names, a list of cutoffs
and the training matrix; every metric is instantiated once per cutoff.
`evaluate` then makes **one pass** over an evaluation loader: for each batch of
users it asks the model for scores, removes the training items, and updates
every metric.

In [5]:
from warprec.evaluation import Evaluator

evaluator = Evaluator(
    ["nDCG", "Recall", "Precision", "HitRate"],
    [10, 20],
    train_set=dataset.train_set.get_sparse(),
)
evaluator.evaluate(models["EASE"], dataset.get_evaluation_dataloader(), "full", dataset)
results = evaluator.compute_results()
{k: list(metrics) for k, metrics in results.items()}

{10: ['nDCG', 'Recall', 'Precision', 'HitRate'],
 20: ['nDCG', 'Recall', 'Precision', 'HitRate']}

The metrics share their intermediate tensors. Each declares the blocks it
needs (the relevance matrix, the top-k indices, the relevance of the top-k…);
the evaluator computes their union once per batch, the top-k once for the
largest cutoff, and slices it for the smaller ones. Adding a metric that
needs no new block costs only its own arithmetic.

In [6]:
pd.DataFrame(
    {
        metric.name: {b.name: "x" for b in metric.components}
        for metric in evaluator.metrics[10]
    }
).fillna("").sort_index()

,nDCG,Recall,Precision,HitRate
BINARY_RELEVANCE,,x,x,x
DISCOUNTED_RELEVANCE,x,,,
TOP_K_BINARY_RELEVANCE,,x,x,x
TOP_K_DISCOUNTED_RELEVANCE,x,,,
VALID_USERS,x,x,x,x


### A per-user vector, and the number a run reports

`compute_results()` returns `{cutoff: {metric: value}}`. A metric averaged over
users does not come back averaged: it is a tensor with **one entry per user
index**, and the number the pipelines log and write is its `nanmean`.

In [7]:
ndcg = results[10]["nDCG"]
print(
    type(ndcg).__name__,
    tuple(ndcg.shape),
    "| first users:",
    [round(v, 4) for v in ndcg[:5].tolist()],
)
print("reported nDCG@10:", round(ndcg.nanmean().item(), 4))

Tensor (943,) | first users: [0.5767, 0.6332, 0.2337, 0.214, 0.0]
reported nDCG@10: 0.2089


Here every user has test items, so no entry is NaN. A user **without** test
items is NaN, not 0: they were not evaluated, and averaging a 0 for them would
pull the score down. To see it, evaluate on the women only: the training data
stays the same, the men's test rows are removed.

In [8]:
from warprec.data import Dataset
from warprec.data.reader import LocalReader
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

data = LocalReader().read_tabular(local_path=str(folder / "ratings.tsv"), sep="\t")
train, _, test = Splitter().split_transaction(
    data, test=SplitSpec(strategy="temporal_holdout", ratio=0.2)
)
women = users.user_id[users.gender == "F"].tolist()
women_only = Dataset(
    train_data=train,
    eval_data=test.filter(nw.col("user_id").is_in(women)),
    rating_type="implicit",
)
ease_w = model_registry.get(
    "EASE",
    params=config.models["EASE"],
    info=women_only.info(),
    interactions=women_only.train_set,
)
evaluator.evaluate(ease_w, women_only.get_evaluation_dataloader(), "full", women_only)
ndcg_w = evaluator.compute_results()[10]["nDCG"]
print(f"{torch.isnan(ndcg_w).sum().item()} of {len(ndcg_w)} users are NaN")
print("nanmean (what WarpRec reports):", round(ndcg_w.nanmean().item(), 4))
print("NaN read as 0 (wrong):         ", round(ndcg_w.nan_to_num(0).mean().item(), 4))

670 of 943 users are NaN
nanmean (what WarpRec reports): 0.2033
NaN read as 0 (wrong):          0.0588


The 670 men are NaN and the reported nDCG@10 is the mean over the 273 women;
counting the men as zeros would report 0.059 instead of 0.203.

Global metrics (`AUC`, `ItemCoverage`, `Gini`, …) are plain numbers instead of
tensors. The section below shows which is which.

### The same evaluator from a configuration

The pipelines never construct the `Evaluator` by hand: `build_evaluator` reads
the `evaluation` section and passes the dataset's feature matrix, clusters and
stash along. From `configs/evaluation.yml` it builds the evaluator above:

In [9]:
from warprec.evaluation import build_evaluator

from_config = build_evaluator(config.evaluation, dataset)
from_config.evaluate(
    models["EASE"], dataset.get_evaluation_dataloader(), "full", dataset
)
same = from_config.compute_results()
print(
    "same metrics:",
    {k: list(v) for k, v in same.items()} == {k: list(v) for k, v in results.items()},
)
print(
    "same values: ",
    all(
        torch.allclose(same[k][m], results[k][m], equal_nan=True)
        for k in results
        for m in results[k]
    ),
)

same metrics: True
same values:  True


A small helper does what the pipelines' log and results file do: evaluate a
model and turn every per-user tensor into its `nanmean`.

In [10]:
def evaluate(evaluator, model, data=dataset, strategy="full", loader=None):
    """Evaluate one model; per-user metrics reported as their nanmean."""
    loader = loader or data.get_evaluation_dataloader()
    evaluator.evaluate(model, loader, strategy, data)
    return {
        f"{metric}@{k}": value.nanmean().item() if torch.is_tensor(value) else value
        for k, metrics in evaluator.compute_results().items()
        for metric, value in metrics.items()
    }

## Full and sampled evaluation

`strategy: full` ranks every item the user has not trained on. `strategy:
sampled` ranks each user's test items against `num_negatives` items drawn
from those the user has neither trained nor been tested on. The draw is
`uniform` by default; `popularity` draws in proportion to the training count
raised to `neg_alpha`, so the negatives are harder to beat.

In [11]:
print(Path("configs/sampled.yml").read_text())

# Rank each user's test items against 99 drawn negatives instead of the catalogue.
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall, Precision, HitRate]
  strategy: sampled
  num_negatives: 99
  negative_sampling: popularity   # or uniform (the default)
  neg_alpha: 0.75                 # read by popularity only
  seed: 42                        # the negatives' draw and the tie-breaking



A fragment like this validates on its own through `EvaluationConfig`, the
model behind the `evaluation` section:

In [12]:
from warprec.utils.config.evaluation_configuration import EvaluationConfig

sampled = EvaluationConfig(
    **yaml.safe_load(Path("configs/sampled.yml").read_text())["evaluation"]
)
(
    sampled.strategy,
    sampled.num_negatives,
    sampled.negative_sampling,
    sampled.neg_alpha,
    sampled.seed,
)

('sampled', 99, 'popularity', 0.75, 42)

The sampled loader is built by the dataset and draws its negatives once, from
a stream seeded with its `seed` (in a run, `evaluation.seed`). The same evaluator scores the three models under the
three protocols:

In [13]:
accuracy = Evaluator(
    ["nDCG", "Recall", "HitRate"], [10], train_set=dataset.train_set.get_sparse()
)
loaders = {
    "full": ("full", None),
    "sampled, uniform": (
        "sampled",
        dataset.get_sampled_evaluation_dataloader(num_negatives=99, seed=42),
    ),
    "sampled, popularity": (
        "sampled",
        dataset.get_sampled_evaluation_dataloader(
            num_negatives=99, seed=42, negative_sampling="popularity", neg_alpha=0.75
        ),
    ),
}
protocols = pd.DataFrame(
    {
        (protocol, name): evaluate(accuracy, model, strategy=strategy, loader=loader)
        for protocol, (strategy, loader) in loaders.items()
        for name, model in models.items()
    }
).T
protocols.round(3)

nDCG@10  Recall@10  HitRate@10
full                Pop        0.122      0.066       0.564
                    ItemKNN    0.196      0.120       0.709
                    EASE       0.209      0.125       0.740
sampled, uniform    Pop        0.488      0.289       0.938
                    ItemKNN    0.644      0.409       0.995
                    EASE       0.666      0.435       0.998
sampled, popularity Pop        0.233      0.137       0.785
                    ItemKNN    0.384      0.249       0.930
                    EASE       0.437      0.296       0.957

Against 99 uniform negatives nDCG and Recall rise three- to fourfold (EASE's
nDCG@10 goes from 0.209 to 0.666) and HitRate@10 all but saturates (0.998).
Popularity negatives are harder to beat, and pull the numbers back down
(0.437). The order of the three models is the same under all three
protocols, but the gaps are not: EASE leads ItemKNN by 0.013 nDCG@10 in full
evaluation, 0.022 against uniform negatives and 0.053 against popularity
negatives. A sampled number is
comparable only with one drawn the same way: same `num_negatives`, same
`negative_sampling` and `neg_alpha`, same seed.

## `mask_seen`

Under `full`, the evaluator sets the score of every item the user trained on
to −∞ before ranking. `mask_seen` decides which:

- `pair`: every item the user has interacted with in training;
- `context`: only items seen **in the same context**, for datasets with
  contextual columns (guide 5);
- `auto` (default): `context` when the dataset has contextual columns,
  `pair` otherwise;
- `none`: nothing; training items compete with test items.

Sampled evaluation never draws a training item, so it ignores the key.

In [14]:
print(Path("configs/mask-seen.yml").read_text())

# Leave training items in the ranking (full strategy only).
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall, Precision, HitRate]
  mask_seen: none   # auto (default), pair, context or none



In [15]:
masking = {}
for policy in ["auto", "pair", "none"]:
    evaluator = Evaluator(
        ["nDCG", "Recall"],
        [10],
        train_set=dataset.train_set.get_sparse(),
        mask_seen=policy,
    )
    for name, model in models.items():
        masking[(name, policy)] = evaluate(evaluator, model)
pd.DataFrame(masking).T.unstack().reindex(
    index=list(models), columns=["auto", "pair", "none"], level=1
).round(4)

nDCG@10                 Recall@10                
           auto    pair    none      auto    pair    none
Pop      0.1218  0.1218  0.0571    0.0658  0.0658  0.0388
ItemKNN  0.1958  0.1958  0.0505    0.1199  0.1199  0.0451
EASE     0.2089  0.2089  0.0264    0.1254  0.1254  0.0309

With no dataset contexts, `auto` is `pair`, so the two columns agree. With
`none` every model collapses, and the order flips: Pop, which ignores the
user's history, now scores the highest nDCG@10 and EASE the lowest. The
models that reproduce their training data best are the ones the unmasked
ranking punishes most, because their lists fill up with it:

In [16]:
train_matrix = dataset.train_set.get_sparse()
seen = torch.from_numpy(train_matrix.toarray()).bool()
all_users = torch.arange(n_users)
for name, model in models.items():
    with torch.inference_mode():
        top10 = model.predict(user_indices=all_users).topk(10, dim=1).indices
    share = seen.gather(1, top10).float().mean().item()
    print(f"{name:8s} unmasked top 10: {share:.0%} training items")

Pop      unmasked top 10: 47% training items
ItemKNN  unmasked top 10: 70% training items
EASE     unmasked top 10: 85% training items


## A tour of the metric families

WarpRec registers its metrics by name (case-insensitive). Most take no
parameter and go under `metrics`; those that do go under `complex_metrics`,
as a `name` and a `params` mapping. A metric given non-default parameters is
named after them, so several variants of one metric can sit side by side. The configuration below takes one or two from each family:

In [17]:
print(Path("configs/metrics.yml").read_text())

# One metric or two from every family. Metrics that need parameters go under
# complex_metrics; the others under metrics.
evaluation:
  top_k: [10]
  metrics:
    [nDCG, Recall, Precision, HitRate, MAP, MRR, AUC, GAUC,  # accuracy
     ItemCoverage, UserCoverage,                              # coverage
     EPC, EFD,                                                # novelty
     Gini, ShannonEntropy, SRecall,                           # diversity (SRecall: side info)
     ARP, APLT, PopREO,                                       # popularity bias
     UserMADRanking, BiasDisparityBS, BiasDisparityBR, BiasDisparityBD]  # fairness: clusters
  complex_metrics:
    - name: F1                        # F-beta of any two metrics
      params: {metric_1: nDCG, metric_2: MAP, beta: 0.5}
    - name: EPC                       # graded instead of binary relevance
      params: {relevance: discounted}
    - name: APLT                      # long tail = items outside the top 50% of interactions
      

A metric receives everything it may need when it is built: the training
matrix and the interaction count of every item (popularity, long tail,
novelty), the feature matrix (`SRecall`), the user and item clusters (the
fairness metrics) and the stash. `build_evaluator` passes all of them, so the
configuration needs nothing more than the names.

In [18]:
tour_config = EvaluationConfig(
    **yaml.safe_load(Path("configs/metrics.yml").read_text())["evaluation"]
)
tour = build_evaluator(tour_config, dataset)
tour_results = {}
for name, model in models.items():
    tour.evaluate(model, dataset.get_evaluation_dataloader(), "full", dataset)
    tour_results[name] = tour.compute_results()[10]


def family(*metrics):
    """One row per metric, one column per model; per-user metrics as nanmean."""
    return pd.DataFrame(
        {
            name: {
                m: v.nanmean().item() if torch.is_tensor(v) else v
                for m, v in res.items()
                if m in metrics
            }
            for name, res in tour_results.items()
        }
    ).loc[list(metrics)]


def kind(value):
    return f"per user {tuple(value.shape)}" if torch.is_tensor(value) else "global"


pd.Series({m: kind(v) for m, v in tour_results["EASE"].items() if "Bias" not in m})

nDCG                                            per user (943,)
Recall                                          per user (943,)
Precision                                       per user (943,)
HitRate                                         per user (943,)
MAP                                             per user (943,)
MRR                                             per user (943,)
AUC                                                      global
GAUC                                            per user (943,)
ItemCoverage                                             global
UserCoverage                                             global
EPC                                             per user (943,)
EFD                                             per user (943,)
Gini                                                     global
ShannonEntropy                                           global
SRecall                                         per user (943,)
ARP                                     

### Accuracy

`AUC` is global: one area over all positive–negative pairs. `GAUC` computes
the area per user and averages, so it is a per-user vector. Neither depends on
the cutoff: they are computed again at every cutoff with the same value. `F1`
with `metric_1`/`metric_2` combines any two metrics, here an F0.5 of nDCG and
MAP, and is named after them.

In [19]:
family(
    "nDCG",
    "Recall",
    "Precision",
    "HitRate",
    "MAP",
    "MRR",
    "AUC",
    "GAUC",
    "F1[nDCG, MAP]",
).round(4)

,Pop,ItemKNN,EASE
nDCG,0.1218,0.1958,0.2089
Recall,0.0658,0.1199,0.1254
Precision,0.1094,0.1699,0.1815
HitRate,0.5642,0.7094,0.7402
MAP,0.0562,0.1011,0.1106
MRR,0.2496,0.3626,0.3805
AUC,0.7852,0.7992,0.8280
GAUC,0.8081,0.8396,0.8579
"F1[nDCG, MAP]",0.0958,0.1292,0.1443


### Coverage, novelty, diversity

`ItemCoverage` counts the distinct items recommended to anyone;
`UserCoverage` counts the users that were evaluated (those with test items).
`EPC` and `EFD` reward hits on unpopular items. `relevance: discounted` swaps
the binary hit for the graded gain nDCG uses; on implicit data every hit has
the same grade, so `EPC[discounted]` is exactly three times `EPC`. `Gini` and `ShannonEntropy` describe how concentrated
the recommendations are over the catalogue; `SRecall` is the share of the
genres among a user's test items that the hits in the top 10 cover.

In [20]:
family(
    "ItemCoverage",
    "UserCoverage",
    "EPC",
    "EPC[discounted]",
    "EFD",
    "Gini",
    "ShannonEntropy",
    "SRecall",
).round(4)

,Pop,ItemKNN,EASE
ItemCoverage,72.0000,207.0000,443.0000
UserCoverage,943.0000,943.0000,943.0000
EPC,0.0662,0.1261,0.1402
EPC[discounted],0.1987,0.3782,0.4206
EFD,0.8796,1.4816,1.6279
Gini,0.9873,0.9558,0.9076
ShannonEntropy,3.3346,4.5744,5.3049
SRecall,0.2136,0.3152,0.3156


### Popularity bias

`ARP` is the mean training count of the recommended items. `APLT` is the share
of each list in the long tail, where the long tail is what is left after the
most popular items that make up `pop_ratio` (default 0.8) of the
interactions; `APLT[Pop50%]` moves that line. `PopREO` compares how often the
head's and the tail's relevant items are retrieved: the standard deviation
of the two retrieval rates over their mean, so 0 is parity and 1 means one of
the two groups is never retrieved.

In [21]:
family("ARP", "APLT", "APLT[Pop50%]", "PopREO").round(4)

,Pop,ItemKNN,EASE
ARP,385.8584,275.9980,245.0712
APLT,0.0000,0.0002,0.0064
APLT[Pop50%],0.0000,0.0155,0.1080
PopREO,1.0000,0.9956,0.9570


With the default line the long tail is out of reach of all three models: even
EASE puts 0.6% of its lists there. Drawn at 50% of the interactions, the line
separates them (Pop 0%, ItemKNN 1.6%, EASE 10.8%). Pop never retrieves a
relevant long-tail item, hence its `PopREO` of 1.

### Fairness

The fairness metrics compare groups, so they read the clusters. Cluster
labels are renumbered from 1 in sorted order (guide 4): here women are user
cluster 1, men cluster 2, and the genres are numbered in sorted order.
`UserMADRanking` is the mean absolute difference between the groups' average
nDCG:

In [22]:
family("UserMADRanking").round(4)

,Pop,ItemKNN,EASE
UserMADRanking,0.027,0.0197,0.008


The gap between women's and men's nDCG@10 is smallest for EASE (0.008) and
largest for Pop (0.027).
The bias-disparity metrics return one value per user cluster × item cluster
pair, named `…_UC<u>_IC<i>`. `BiasDisparityBS` describes the training data: a
group's share of interactions on a genre over the genre's share of the
catalogue. `BiasDisparityBR` is the same ratio on the recommendations, and
`BiasDisparityBD` the relative change from one to the other, so a negative
value means the model under-recommends that genre to that group compared with
what the group consumed. For EASE and three genres:

In [23]:
user_codes = {1: "F", 2: "M"}
item_codes = {
    int(code): label
    for code, label in zip(
        dataset.get_item_cluster().tolist(),
        [dataset.item_cluster.get(i) for i in range(n_items)],
    )
}
rows = []
for gender_code, gender in user_codes.items():
    for genre in ["Action", "Comedy", "Drama"]:
        genre_code = next(c for c, g in item_codes.items() if g == genre)
        key = f"UC{gender_code}_IC{genre_code}"
        rows.append(
            {
                "users": gender,
                "first genre": genre,
                **{
                    m: tour_results["EASE"][f"BiasDisparity{m}_{key}"]
                    for m in ["BS", "BR", "BD"]
                },
            }
        )
pd.DataFrame(rows).round(3)

,users,first genre,BS,BR,BD
0,F,Action,1.413,1.917,0.810
1,F,Comedy,0.940,0.704,-0.398
2,F,Drama,0.986,0.992,-0.192
3,M,Action,1.770,2.276,0.522
4,M,Comedy,0.846,0.629,-0.329
5,M,Drama,0.871,0.919,-0.108


Women and men both consume more Action than its share of the catalogue
(`BS` above 1); EASE amplifies that for both groups (`BD` +0.81 and +0.52) and
recommends Comedy less than either group watched it (−0.40 and −0.33).

### Several objectives at once

`EucDistance` and `Hypervolume` combine metrics per user: the distance from a
utopia point, and the volume dominated over a nadir point. They take their
components by name, and both are named after their parameters.

In [24]:
family(
    "EucDistance[nDCG, EPC][1.0, 1.0]",
    "Hypervolume[nDCG, EPC][0.0, 0.0][true, true]",
).round(4)

,Pop,ItemKNN,EASE
"EucDistance[nDCG, EPC][1.0, 1.0]",1.2836,1.2361,1.214
"Hypervolume[nDCG, EPC][0.0, 0.0][true, true]",0.0216,0.0295,0.035


### Rating metrics

`RMSE`, `MAE` and `MSE` compare a model's score with the rating, on the test
pairs only. They need two things: a dataset read as `explicit`, so the ground
truth holds ratings rather than 1s; and scores on the rating scale, which
none of the ranking models above produces. As a stand-in, a bias baseline
(global mean + user offset + item offset, from the training split) is
written to a file and read through `ProxyRecommender`, the model that
evaluates a file of predictions (guide 14):

In [25]:
train_pd = train.to_pandas()
mu = train_pd.rating.mean()
item_bias = (train_pd.rating - mu).groupby(train_pd.item_id).sum() / (
    train_pd.groupby("item_id").size() + 10
)
residual = train_pd.rating - mu - train_pd.item_id.map(item_bias)
user_bias = residual.groupby(train_pd.user_id).sum() / (
    train_pd.groupby("user_id").size() + 10
)

test_pd = test.to_pandas()
test_pd["rating"] = (
    mu
    + test_pd.user_id.map(user_bias).fillna(0)
    + test_pd.item_id.map(item_bias).fillna(0)
)
test_pd[["user_id", "item_id", "rating"]].to_csv(
    RUNS / "baseline-predictions.tsv", sep="\t", index=False
)

explicit = Dataset(
    train_data=train, eval_data=test, rating_type="explicit", rating_label="rating"
)
baseline = model_registry.get(
    "ProxyRecommender",
    params={
        "recommendation_file": str(RUNS / "baseline-predictions.tsv"),
        "separator": "\t",
        "header": True,
    },
    info=explicit.info(),
)
ease_explicit = model_registry.get(
    "EASE",
    params=config.models["EASE"],
    info=explicit.info(),
    interactions=explicit.train_set,
)
rating = Evaluator(["RMSE", "MAE"], [10], train_set=explicit.train_set.get_sparse())
pd.DataFrame(
    {
        "bias baseline": evaluate(rating, baseline, explicit),
        "EASE": evaluate(rating, ease_explicit, explicit),
    }
).round(3)

,bias baseline,EASE
RMSE@10,0.973,3.264
MAE@10,0.817,3.115


The baseline is off by 0.973 stars in RMSE. EASE's 3.264 measures nothing
useful: its scores are item–item weights, not ratings. The `@10` in the names
is only the cutoff the evaluator was built with; rating metrics ignore it.

## Per-user results in a run

Since every per-user metric is kept per user, a run can write it out:
`save_per_user: true` makes the train and eval pipelines write one file per
model and cutoff next to the overall results, with a row per user ID and a
column per per-user metric. Global metrics are left out of it. The writer call
behind it:

In [26]:
from warprec.data.writer import LocalWriter

writer = LocalWriter(
    dataset_name="ml-100k", local_path=str(RUNS / "written"), timestamp="guide"
)
user_ids, _ = women_only.get_inverse_mappings()
evaluator = Evaluator(
    ["nDCG", "Recall", "ItemCoverage"],
    [10],
    train_set=women_only.train_set.get_sparse(),
)
evaluator.evaluate(ease_w, women_only.get_evaluation_dataloader(), "full", women_only)
writer.write_results_per_user(evaluator.compute_results(), "EASE", user_ids)
written = sorted(Path(writer.experiment_evaluation_path).glob("*per_user*"))
print([p.name for p in written])
pd.read_csv(written[0], sep="\t").head()

['EASE_k_10_per_user_guide.tsv']


,user_id,nDCG,Recall
0,1,NaN,NaN
1,2,0.63322,0.384615
2,3,NaN,NaN
3,4,NaN,NaN
4,5,0.00000,0.000000


Users with no test items keep their row, with empty (NaN) values, so the file
lines up with the user list.

Two more keys decide what is evaluated *during* training (guide 10):

- `validation_metric` (`metric@k`, default `nDCG@10`) is the score every
  hyperparameter trial reports and is selected on, and the one early stopping
  watches.
- `full_evaluation_on_report`: by default each report computes the validation
  metric alone; `true` computes the whole `metrics` list at every report,
  which costs time but shows every metric's trajectory. It then requires
  the validation metric and its cutoff to be among `metrics` and `top_k`.

In [27]:
print(EvaluationConfig(top_k=[10], metrics=["nDCG"]).validation_metric)
try:
    EvaluationConfig(top_k=[10], metrics=["nDCG"], validation_metric="nDCG")
except ValueError as error:
    print(str(error).splitlines()[-2].strip())

nDCG@10
Value error, Validation metric nDCG not valid. Validation metric should be defined as: metric_name@top_k. [type=value_error, input_value='nDCG', input_type=str]


## Statistical significance

Per-user results are paired samples: the same users, scored by two models.
`compute_paired_statistical_test` takes `{model: compute_results()}` and runs
one test for every pair of models, every per-user metric and every cutoff.
Global metrics are skipped, and users that are NaN for either model are
dropped from that comparison.

The `stat_significance` section enables the tests; the pipelines run each
enabled test once all models are evaluated, with the corrections applied to
every table. This is the complete configuration, which also loads as a whole:

In [28]:
print(Path("configs/complete.yml").read_text())

# The whole experiment: data, models, and an evaluation section that also asks
# for per-user files and significance tests between every pair of models.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  Pop: {}
  ItemKNN:
    k: 50
    similarity: cosine
  EASE:
    l2: 200.0
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/12-evaluation/experiment
evaluation:
  top_k: [10, 20]
  metrics: [nDCG, Recall, HitRate, ItemCoverage]
  validation_metric: nDCG@10        # what HPO selects on (guide 10)
  full_evaluation_on_report: false  # true: every metric on every report
  save_per_user: true
  stat_significance:
    wilcoxon_test: true
    paired_t_test: true
    corrections:
      bonferroni: true
      holm_bonferroni: true
      fdr: true
      alpha: 0.05



Its models and dataset are the ones above without side information or
clusters, so the evaluation runs on the same split, and the numbers match
the earlier tables. That holds for Pop too, which gives many items the same
score: the evaluator breaks a tie with a key hashed from `evaluation.seed`,
the user and the item, so a user's list does not depend on the batch or on
the other cutoffs.

In [29]:
complete = load_design_configuration("configs/complete.yml")
complete_dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(complete), WarpRecCallback(), complete
)
per_model = {}
for name, params in complete.models.items():
    model = model_registry.get(
        name,
        params=params,
        info=complete_dataset.info(),
        interactions=complete_dataset.train_set,
    )
    evaluator = build_evaluator(complete.evaluation, complete_dataset)
    evaluator.evaluate(
        model, complete_dataset.get_evaluation_dataloader(), "full", complete_dataset
    )
    per_model[name] = evaluator.compute_results()
pd.DataFrame(
    {
        name: {m: v.nanmean().item() for m, v in r[10].items() if torch.is_tensor(v)}
        for name, r in per_model.items()
    }
).round(4)

,Pop,ItemKNN,EASE
nDCG,0.1218,0.1958,0.2089
Recall,0.0658,0.1199,0.1254
HitRate,0.5642,0.7094,0.7402


Each enabled test is run as the pipelines run it, with the `corrections`
section passed as keyword arguments:

In [30]:
from warprec.evaluation import compute_paired_statistical_test

significance = complete.evaluation.stat_significance
corrections = significance.corrections.model_dump()
wilcoxon = compute_paired_statistical_test(
    per_model, "wilcoxon_test", backend="pandas", **corrections
).to_native()
print(list(wilcoxon.columns))
wilcoxon.columns = [c.split(" (")[-1].rstrip(")") for c in wilcoxon.columns]
wilcoxon.drop(columns="Statistic").sort_values(
    ["Cutoff", "Metric", "Model A"]
).set_index(["Cutoff", "Metric", "Model A", "Model B"]).round(6)

['Model A', 'Model B', 'Metric', 'Cutoff', 'Statistic', 'p-value', 'Significance (α=0.05)', 'Significance (Bonferroni α=2.78e-03)', 'Significance (Holm-Bonferroni)', 'Significance (FDR)']


p-value           α=0.05  \
Cutoff Metric  Model A Model B                              
10     HitRate ItemKNN EASE     0.037826      Significant   
               Pop     EASE     0.000000      Significant   
                       ItemKNN  0.000000      Significant   
       Recall  ItemKNN EASE     0.070401  Not significant   
               Pop     EASE     0.000000      Significant   
                       ItemKNN  0.000000      Significant   
       nDCG    ItemKNN EASE     0.012389      Significant   
               Pop     EASE     0.000000      Significant   
                       ItemKNN  0.000000      Significant   
20     HitRate ItemKNN EASE     0.064465  Not significant   
               Pop     EASE     0.000000      Significant   
                       ItemKNN  0.000000      Significant   
       Recall  ItemKNN EASE     0.000005      Significant   
               Pop     EASE     0.000000      Significant   
                       ItemKNN  0.000000      Significant   
       nDCG    ItemKNN EASE     0.000025      Significant   
               Pop     ItemKNN  0.000000      Significant   
                       EASE     0.000000      Significant   

                               Bonferroni α=2.78e-03  Holm-Bonferroni  \
Cutoff Metric  Model A Model B                                          
10     HitRate ItemKNN EASE          Not significant  Not significant   
               Pop     EASE              Significant      Significant   
                       ItemKNN           Significant      Significant   
       Recall  ItemKNN EASE          Not significant  Not significant   
               Pop     EASE              Significant      Significant   
                       ItemKNN           Significant      Significant   
       nDCG    ItemKNN EASE          Not significant      Significant   
               Pop     EASE              Significant      Significant   
                       ItemKNN           Significant      Significant   
20     HitRate ItemKNN EASE          Not significant  Not significant   
               Pop     EASE              Significant      Significant   
                       ItemKNN           Significant      Significant   
       Recall  ItemKNN EASE              Significant      Significant   
               Pop     EASE              Significant      Significant   
                       ItemKNN           Significant      Significant   
       nDCG    ItemKNN EASE              Significant      Significant   
               Pop     ItemKNN           Significant      Significant   
                       EASE              Significant      Significant   

                                            FDR  
Cutoff Metric  Model A Model B                   
10     HitRate ItemKNN EASE         Significant  
               Pop     EASE         Significant  
                       ItemKNN      Significant  
       Recall  ItemKNN EASE     Not significant  
               Pop     EASE         Significant  
                       ItemKNN      Significant  
       nDCG    ItemKNN EASE         Significant  
               Pop     EASE         Significant  
                       ItemKNN      Significant  
20     HitRate ItemKNN EASE     Not significant  
               Pop     EASE         Significant  
                       ItemKNN      Significant  
       Recall  ItemKNN EASE         Significant  
               Pop     EASE         Significant  
                       ItemKNN      Significant  
       nDCG    ItemKNN EASE         Significant  
               Pop     ItemKNN      Significant  
                       EASE         Significant

The first line printed is the table as WarpRec writes it; the column names
are shortened for display. Each row is one test, and each significance column
says `Significant` or `Not significant`. `α=0.05` is the uncorrected test;
Bonferroni divides α by the number of rows (18 here, so 2.78e-03).
Holm–Bonferroni sorts the p-values and compares the i-th smallest of m with
α / (m − i + 1), stopping at the first that fails: every row after it is not
significant. FDR (Benjamini–Hochberg) finds the largest rank k whose p-value is
below k·α / m and declares the k smallest significant, including any that
missed its own threshold.

Pop loses to both models everywhere. ItemKNN against EASE is the pair to
read: at nDCG@10 (p = 0.012) the difference passes the uncorrected test,
Holm and FDR, but not Bonferroni; at Recall@10 (p = 0.070) it passes none.

The four tests answer different questions:

| key | test | pairing |
|---|---|---|
| `paired_t_test` | paired t-test on the per-user differences | paired |
| `wilcoxon_test` | Wilcoxon signed-rank | paired |
| `kruskal_test` | Kruskal–Wallis H | treats the two models as independent samples |
| `whitney_u_test` | Mann–Whitney U | treats the two models as independent samples |

Per-user scores of two models on the same users are paired, so the first two
are the ones that fit; the last two ignore the pairing and are less
sensitive. On ItemKNN against EASE at nDCG@10:

In [31]:
pd.DataFrame(
    {
        test: compute_paired_statistical_test(
            {n: per_model[n] for n in ["ItemKNN", "EASE"]}, test, backend="pandas"
        )
        .to_native()
        .set_index(["Metric", "Cutoff"])
        .loc[("nDCG", 10), "p-value"]
        for test in ["paired_t_test", "wilcoxon_test", "kruskal_test", "whitney_u_test"]
    },
    index=["p-value"],
).T

,p-value
paired_t_test,0.008611
wilcoxon_test,0.012389
kruskal_test,0.214997
whitney_u_test,0.215083


## Summary of the `evaluation` section

| key | what it does |
|---|---|
| `top_k` | the cutoffs; every metric is computed at each |
| `metrics` | metric names without parameters |
| `complex_metrics` | `name` + `params` for metrics that take parameters (`F1`, `EPC`/`EFD` `relevance`, long-tail `pop_ratio`, `EucDistance`, `Hypervolume`) |
| `strategy` | `full` (every unseen item) or `sampled` |
| `num_negatives`, `negative_sampling`, `neg_alpha` | how the `sampled` candidates are drawn |
| `seed` | the seed of the sampled negatives and of tie-breaking in the rankings |
| `mask_seen` | `auto`, `pair`, `context` or `none`: which training items are removed from the ranking |
| `batch_size` | users per evaluation batch |
| `validation_metric` | `metric@k` that trials are selected on |
| `full_evaluation_on_report` | compute every metric at every report, not only the validation metric |
| `save_per_user` | also write the per-user metrics, one row per user |
| `max_metric_per_row` | how many metrics each row of the logged results table holds |
| `stat_significance` | `paired_t_test`, `wilcoxon_test`, `kruskal_test`, `whitney_u_test`, and `corrections` (`bonferroni`, `holm_bonferroni`, `fdr`, `alpha`) |

Every key is listed on the [Evaluation configuration](https://warprec.readthedocs.io/en/latest/configuration/evaluation/)
page, and every metric with its formula under
[Metrics](https://warprec.readthedocs.io/en/latest/evaluation/metrics/taxonomy/).
`candidates` and `propensity`, which restrict the ranking to cold items and
correct for exposure, are guide 13's subject.